# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [ ]:
import numpy as np
import pandas as pd
import csv
import json
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50,VGG16,MobileNetV2# (seleccióna la arquitectura que desees, puedes buscar otras) VGG16, ResNet50, MobileNetV2
from pathlib import Path
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()


## Definiciones de modelos

In [2]:
def crear_augmentation():
    """Define transformaciones aleatorias aplicadas solo durante el entrenamiento."""
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.08, 0.08)
    ])

# Funcion de data augmentation para aumentar la variabilidad de los datos de entrenamiento y evitar el sobreajuste del modelo.

#-------------------------------------------------------------------------------

def crear_cnn_secuencial():
    """Construye una CNN secuencial didactica con regularizacion L2 y dropout."""

    # CNN secuencial personalizada.

    entradas = keras.Input(shape=(64, 64, 3))

    x = crear_augmentation()(entradas)
    x = layers.Rescaling(1./255)(x)

    for filtros in [32, 64, 128]:
        x = layers.Conv2D(
            filtros, 3, padding="same",
            activation="relu"
        )(x)

        x = layers.Conv2D(
            filtros, 3, padding="same",
            activation="relu"
        )(x)

        x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(0.3)(x)

    salidas = layers.Dense(
        10, activation="softmax"
    )(x)

    return keras.Model(entradas, salidas)

#---------------------------------------------------------------------------------------

# CNN residual personalizada inspirada en ResNet.

def bloque_residual(x, filtros, stride=1):
    """Aplica dos convoluciones y suma una atajo proyectado si es necesario."""

    atajo = x

    # Primera convolución
    y = layers.Conv2D(
        filtros, 3, strides=stride,
        padding="same", use_bias=False
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    # Segunda convolución
    y = layers.Conv2D(
        filtros, 3,
        padding="same", use_bias=False
    )(y)
    y = layers.BatchNormalization()(y)

    # Proyección cuando cambian las dimensiones
    if stride != 1 or x.shape[-1] != filtros:
        atajo = layers.Conv2D(
            filtros, 1, strides=stride,
            padding="same", use_bias=False
        )(atajo)
        atajo = layers.BatchNormalization()(atajo)

    # Conexión residual
    y = layers.Add()([y, atajo])

    return layers.ReLU()(y)


def crear_cnn_residual():
    """Construye una CNN residual didactica sin modificar la arquitectura base."""

    entradas = keras.Input(shape=(64, 64, 3))

    x = crear_augmentation()(entradas)
    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(
        32, 3, padding="same", use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = bloque_residual(x, 32)
    x = bloque_residual(x, 64, stride=2)
    x = bloque_residual(x, 128, stride=2)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)

    salidas = layers.Dense(
        10, activation="softmax"
    )(x)

    return keras.Model(entradas, salidas)

#---------------------------------------------------------------------------------------

# MobileNetV2 con transfer learning.

def crear_mobilenet_transfer():
    """Construye MobileNetV2 congelada y agrega una cabeza para CIFAR-10."""

    base = keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(64, 64, 3)
    )

    base.trainable = False

    entradas = keras.Input(shape=(64, 64, 3))

    x = crear_augmentation()(entradas)

    # Preprocesamiento esperado por MobileNetV2
    x = layers.Rescaling(
        scale=1./127.5,
        offset=-1
    )(x)

    x = base(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)

    salidas = layers.Dense(
        10, activation="softmax"
    )(x)

    return keras.Model(entradas, salidas)

In [3]:
cnn_1 = crear_cnn_secuencial()
cnn_2 = crear_cnn_residual()
cnn_3 = crear_mobilenet_transfer()

cnn_1.summary()
cnn_2.summary()
cnn_3.summary(expand_nested=True)



9406464/9406464 [==============================] - 0s 0us/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 64, 64, 3)]       0         
                                                                 
 sequential (Sequential)     (None, 64, 64, 3)         0         
                                                                 
 rescaling (Rescaling)       (None, 64, 64, 3)         0         
                                                                 
 conv2d (Conv2D)             (None, 64, 64, 32)        896       
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 64, 32)        9248      
                                                                 
 max_pooling2d (MaxPooling2  (None, 32, 32, 32)        0         
 D)                                                           

## Entrenamiento de modelos.

In [ ]:
# Funcion para preparar los datos de entrenamiento, validacion y prueba.
# Los modelos realizan su propio escalado/preprocesamiento dentro de la arquitectura.

def preparar_datos(
    imagenes,
    etiquetas,
    arquitectura,
    batch_size=32,
    entrenamiento=False
):
    """Crea un dataset tf.data con resize, etiquetas one-hot y batching comun."""

    # Las imagenes llegan en RGB con valores [0, 255].
    if arquitectura not in {"vgg16", "resnet50", "mobilenet"}:
        raise ValueError("Arquitectura no reconocida")

    dataset = tf.data.Dataset.from_tensor_slices(
        (imagenes, etiquetas)
    )

    if entrenamiento:
        dataset = dataset.shuffle(
            buffer_size=10000,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(batch_size)

    def transformar(x, y):
        # Redimensionar CIFAR-10 de 32x32 a 64x64.
        x = tf.image.resize(
            tf.cast(x, tf.float32),
            [64, 64]
        )

        # Cada modelo aplica su escalado dentro de la arquitectura:
        # Rescaling(1./255) o Rescaling(1./127.5, offset=-1).
        y = tf.cast(tf.reshape(y, [-1]), tf.int32)
        # Convertir cada etiqueta entera (0-9) en un vector de 10 posiciones.
        # Esto es compatible con categorical_crossentropy y softmax.
        y = tf.one_hot(y, depth=10)
        return x, y

    dataset = dataset.map(
        transformar,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return dataset.prefetch(tf.data.AUTOTUNE)


In [ ]:
"""
Entrenamiento reproducible de una arquitectura CNN seleccionada.

Ejecutar esta celda una vez por arquitectura que se necesite entrenar. Los
hiperparametros y la particion se mantienen constantes para la comparacion.
"""

# Protocolo comun de comparacion.
SEED = 42

# Cargar CIFAR-10.
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Separar 5,000 ejemplos de validacion conservando la proporcion de las 10 clases.
x_fit, x_val, y_fit, y_val = train_test_split(
    x_train,
    y_train,
    test_size=5000,
    random_state=SEED,
    stratify=y_train.reshape(-1)
)

ARQUITECTURA = "vgg16"  # Usar tambien "resnet50" o "mobilenet".
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 15

# Directorios destinados a los artefactos generados por los experimentos.
ARTEFACTOS_DIR = Path("artefactos")
MODELOS_DIR = ARTEFACTOS_DIR / "modelos"
HISTORIALES_DIR = ARTEFACTOS_DIR / "historiales"
CONFIGURACIONES_DIR = ARTEFACTOS_DIR / "configuraciones"
COMPARACIONES_DIR = ARTEFACTOS_DIR / "comparaciones"

for directorio in (
    MODELOS_DIR,
    HISTORIALES_DIR,
    CONFIGURACIONES_DIR,
    COMPARACIONES_DIR
):
    directorio.mkdir(parents=True, exist_ok=True)

creadores = {
    "vgg16": crear_cnn_secuencial,
    "resnet50": crear_cnn_residual,
    "mobilenet": crear_mobilenet_transfer
}

# Restablecer explicitamente el estado aleatorio antes de crear cada experimento.
# Esto hace comparable la inicializacion; configurar la semilla una sola vez
# no la restablece automaticamente antes de cada nuevo modelo.
keras.utils.set_random_seed(SEED)
modelo = creadores[ARQUITECTURA]()

train_ds = preparar_datos(
    x_fit, y_fit,
    ARQUITECTURA,
    batch_size=BATCH_SIZE,
    entrenamiento=True
)

val_ds = preparar_datos(
    x_val, y_val,
    ARQUITECTURA,
    batch_size=BATCH_SIZE
)

test_ds = preparar_datos(
    x_test, y_test,
    ARQUITECTURA,
    batch_size=BATCH_SIZE
)

modelo.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELOS_DIR / f"{ARQUITECTURA}_mejor.keras"),
        monitor="val_loss",
        mode="min",
        save_best_only=True
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True
    ),
    keras.callbacks.CSVLogger(
        str(HISTORIALES_DIR / f"{ARQUITECTURA}_historial.csv")
    )
]

historia = modelo.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Guardar tambien el modelo con los mejores pesos restaurados.
modelo.save(str(MODELOS_DIR / f"{ARQUITECTURA}_cifar10.keras"))

# Persistir la configuracion para reproducir y auditar la ejecucion.
configuracion = {
    "arquitectura": ARQUITECTURA,
    "seed": SEED,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs_maximas": EPOCHS,
    "validation_size": 5000,
    "split": "estratificado",
    "input_shape": [64, 64, 3],
    "num_classes": 10,
    "preprocesamiento": "escalado interno de la arquitectura"
}
with open(CONFIGURACIONES_DIR / f"{ARQUITECTURA}_configuracion.json", "w", encoding="utf-8") as archivo:
    json.dump(configuracion, archivo, indent=2)


In [ ]:
from pathlib import Path
from tensorflow import keras

ruta = Path("artefactos") / "modelos" / "resnet50_cifar10.keras"
if ruta.exists():
    modelo_cargado_resnet = keras.models.load_model(ruta)
    print("Modelo cargado:", ruta)
    print("Entrada:", modelo_cargado_resnet.input_shape)
    print("Salida:", modelo_cargado_resnet.output_shape)
    print("Parametros:", modelo_cargado_resnet.count_params())
else:
        print(f"Aun no existe: {ruta}. Ejecuta primero el entrenamiento de esta arquitectura.")


In [ ]:
from pathlib import Path
from tensorflow import keras

ruta = Path("artefactos") / "modelos" / "vgg16_cifar10.keras"
if ruta.exists():
    modelo_cargado_vgg = keras.models.load_model(ruta)
    print("Modelo cargado:", ruta)
    print("Entrada:", modelo_cargado_vgg.input_shape)
    print("Salida:", modelo_cargado_vgg.output_shape)
    print("Parametros:", modelo_cargado_vgg.count_params())
else:
        print(f"Aun no existe: {ruta}. Ejecuta primero el entrenamiento de esta arquitectura.")


In [ ]:
from pathlib import Path
from tensorflow import keras

ruta = Path("artefactos") / "modelos" / "mobilenet_cifar10.keras"
if ruta.exists():
    modelo_cargado_mobilenet = keras.models.load_model(ruta)
    print("Modelo cargado:", ruta)
    print("Entrada:", modelo_cargado_mobilenet.input_shape)
    print("Salida:", modelo_cargado_mobilenet.output_shape)
    print("Parametros:", modelo_cargado_mobilenet.count_params())
else:
        print(f"Aun no existe: {ruta}. Ejecuta primero el entrenamiento de esta arquitectura.")


In [ ]:
from pathlib import Path
import zipfile
from tensorflow import keras

ruta = Path("artefactos") / "modelos" / "resnet50_cifar10.keras"

if not ruta.exists():
    print(f"Aun no existe {ruta}. Ejecuta el entrenamiento de esta arquitectura para generarlo.")
else:
    print("Existe:", ruta.exists())
    print("Tamano:", ruta.stat().st_size, "bytes")
    with zipfile.ZipFile(ruta) as archivo:
        print("Archivo comprimido valido:", archivo.testzip() is None)
    modelo = keras.models.load_model(ruta, compile=False)
    print("Carga correcta")
    print("Entrada:", modelo.input_shape)
    print("Salida:", modelo.output_shape)
    print("Parametros:", modelo.count_params())


In [ ]:
nombres = [
    "vgg16",
    "resnet50",
    "mobilenet"
]

if "x_test" not in globals():
    (_, _), (x_test, y_test) = keras.datasets.cifar10.load_data()
if "BATCH_SIZE" not in globals():
    BATCH_SIZE = 32

resultados_finales = []

for nombre in nombres:
    ruta = Path("artefactos") / "modelos" / f"{nombre}_cifar10.keras"

    if not ruta.exists():
        print(f"No se encontro: {ruta}")
        continue

    modelo = keras.models.load_model(ruta)
    test_ds = preparar_datos(
        x_test,
        y_test,
        nombre,
        batch_size=BATCH_SIZE
    )

    loss, accuracy = modelo.evaluate(
        test_ds,
        verbose=0
    )

    resultados_finales.append({
        "modelo": nombre,
        "test_loss": float(loss),
        "test_accuracy": float(accuracy),
        "parametros": int(modelo.count_params())
    })

if resultados_finales:
    with open(Path("artefactos") / "comparaciones" / "comparacion.csv", "w", newline="") as archivo:
        writer = csv.DictWriter(
            archivo,
            fieldnames=resultados_finales[0].keys()
        )
        writer.writeheader()
        writer.writerows(resultados_finales)

    with open(Path("artefactos") / "comparaciones" / "comparacion.json", "w", encoding="utf-8") as archivo:
        json.dump(resultados_finales, archivo, indent=2)

for resultado in resultados_finales:
    print(resultado)


## Estadística y gráficos

In [ ]:
# Graficas del historial persistido de la arquitectura seleccionada.
historial_path = Path("artefactos") / "historiales" / f"{ARQUITECTURA}_historial.csv"
if historial_path.exists():
    historial_df = pd.read_csv(historial_path)
elif "historia" in globals():
    historial_df = pd.DataFrame(historia.history)
else:
    historial_df = None

if historial_df is not None:
    epocas = range(1, len(historial_df) + 1)
    figura, ejes = plt.subplots(1, 2, figsize=(12, 4))

    ejes[0].plot(epocas, historial_df["loss"], label="Entrenamiento")
    ejes[0].plot(epocas, historial_df["val_loss"], label="Validacion")
    ejes[0].set_title(f"Perdida: {ARQUITECTURA}")
    ejes[0].set_xlabel("Epoca")
    ejes[0].set_ylabel("Loss")
    ejes[0].legend()

    ejes[1].plot(epocas, historial_df["accuracy"], label="Entrenamiento")
    ejes[1].plot(epocas, historial_df["val_accuracy"], label="Validacion")
    ejes[1].set_title(f"Accuracy: {ARQUITECTURA}")
    ejes[1].set_xlabel("Epoca")
    ejes[1].set_ylabel("Accuracy")
    ejes[1].legend()

    figura.tight_layout()
    plt.show()
else:
    print(f"No existe el historial de {ARQUITECTURA}; ejecuta primero su entrenamiento.")

# Comparacion de modelos disponibles en el archivo persistido.
comparacion_path = Path("artefactos") / "comparaciones" / "comparacion.csv"
if comparacion_path.exists():
    comparacion = pd.read_csv(comparacion_path)
    comparacion.plot(
        x="modelo",
        y="test_accuracy",
        kind="bar",
        legend=False,
        title="Accuracy en el conjunto de prueba"
    )
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print(f"Ejecuta la celda de evaluacion para generar {comparacion_path}.")


# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejorn ¿Por quén ¿Qué mejorarían ¿Cómo lo mejorarían